In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the get_vix() function
def get_vix(price, slow, fast, smooth):
    exp1 = price.ewm(span=fast, adjust=False).mean()
    exp2 = price.ewm(span=slow, adjust=False).mean()
    vix = pd.DataFrame(exp1 - exp2, columns=['vix'])
    signal = pd.DataFrame(vix.ewm(span=smooth, adjust=False).mean(), columns=['signal'])
    hist = pd.DataFrame(vix['vix'] - signal['signal'], columns=['hist'])
    df = pd.concat([vix, signal, hist], axis=1)
    return df

# Define the implement_vix_strategy() function
def implement_vix_strategy(prices, data):
    buy_price = []
    sell_price = []
    vix_signal = []
    no_operation = []
    no_operation_buy = []
    no_operation_sell = []
    signal = -1
    for i in range(len(data)):
        if data['predicted_value'][i] > 0:
            if signal != 1:
                buy_price.append(prices[i])
                sell_price.append(np.nan)
                signal = 1
                vix_signal.append(signal)
                no_operation.append(np.nan)
                no_operation_buy.append(np.nan)
                no_operation_sell.append(np.nan)
            else:
                buy_price.append(np.nan)
                sell_price.append(np.nan)
                vix_signal.append(0)
                no_operation.append(np.nan)
                no_operation_buy.append(prices[i])
                no_operation_sell.append(np.nan)
        elif data['predicted_value'][i] < 0:
            if signal != -1:
                buy_price.append(np.nan)
                sell_price.append(prices[i])
                signal = -1
                vix_signal.append(signal)
                no_operation.append(np.nan)
                no_operation_buy.append(np.nan)
                no_operation_sell.append(np.nan)
            else:
                buy_price.append(np.nan)
                sell_price.append(np.nan)
                vix_signal.append(0)
                no_operation.append(np.nan)
                no_operation_buy.append(np.nan)
                no_operation_sell.append(prices[i])

    return buy_price, sell_price, vix_signal, no_operation, no_operation_buy, no_operation_sell



In [ ]:
DROP = False
%run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Init/Initialization.ipynb"
start_date = "2020-06-01"

# Load the data and prepare it for the plot
df_prediction = pd.DataFrame(data=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Predictions/predictions_ml.pkl')).rename(columns={0:"predicted_value"})
df = df.loc['2020-05-31':'2022-06-01']
df = pd.merge(df_prediction['predicted_value'].diff().dropna(), get_vix(df['VIX'], 26, 12, 9), left_index=True, right_index=True)
df.columns = ['direction', 'vix', 'signal', 'hist']

# Implement the trading strategy and plot the results
buy_price, sell_price, vix_signal, no_operation, no_operation_buy, no_operation_sell = implement_vix_strategy(df['vix'], df)
df['Buy'] = buy_price
df['Sell'] = sell_price
df['VIX Signal'] = vix_signal
df['No operation'] = no_operation
df['No operation Buy'] = no_operation_buy
df['No operation Sell'] = no_operation_sell

fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))

# plot on ax1
ax1.plot(x, y1, color='blue')
ax1.set_xlabel('x')
ax1.set_ylabel('y1')
ax1.set_title('Plot of y1 vs. x')

# plot on ax2
ax2.plot(x, y2, color='red')
ax2.set_xlabel('x')
ax2.set_ylabel('y2')
ax2.set_title('Plot of y2 vs. x')

# add a common title for the entire figure
fig.suptitle('Comparing y1 and y2')

# display the plot
plt.show()
